In [1]:
# This is a transformer from scratch (mostly for fun)

In [2]:

#imports

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

import os
import sys
import random

import typing

torch.set_default_device('cuda')

In [3]:
# data preperation 
# importantly, I am chose to only look at individual characters (rather than tokens) to make the transformer far simpler to implement (no need for a tokenizer)
# Moreover, by focusing on only characters, each character can just be simply assigned an integer, making the code simpler
text = open("trainingdata.txt", "r").read() # should be simple plain text file

chars = sorted(list(set(text)))
string_to_output_ints = {character: i for i, character in enumerate(chars)}
int_to_output_string = {i: character for i, character in enumerate(chars)}

encode = lambda s: [string_to_output_ints[c] for c in s] # encoder that converts any string into int based on training data
decode = lambda ids: "".join([int_to_output_string[i] for i in ids]) # basically the opposite of the encoder

data = torch.tensor(encode(text), dtype=torch.long) # takes all the training data and coverts into into a tensor; uses a datatype of torch.long to hold a signed 64 bit integer (int64)


In [4]:
# Mock up of the actual problem
# since the problem itself is "next token prediction" with each token being an individual character
# each of the inputs must be an array of chars, while the output is a char
# essentially if x = [list of chars] = the input, then y = [list of chars + new_char] = the output
# a good mathematical representation for this can be P(x_(t+1) | x_1, ... , x_t)

In [ ]:
# token embedding works by creating a mathematical space (a vector space) with n_dimensions
# this vector space then houses all the information of each of the chars (tokens)
# This therefore converts the dictionary, which is a simplification done on the code, into a 
# sound vector space that the model can train on
vocab_size = len(chars) # number of unique chars that are present in the text
d_model = 384 # the number of dimensions each vector can have to represent each char
batch_size = 64
max_iters = 4400
eval_interval = 200
learning_rate = 5e-4
eval_iters = 20
num_heads = 6
num_layers = 6
dropout = 0.1
temperature = 0.0001
BLOCK_SIZE: int = 256 # max sequence/context length
device: str = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
token_embeddings = nn.Embedding(vocab_size, d_model) # creates the actual embedding space


# Input:
# B = 4 sequences
# T = 10 tokens per sequence
x = torch.randint(0, vocab_size, (4, 10))

cuda


In [6]:
# positional embedding helps the attention mechanism understand the location at which the token is located in
# This works with regular token embedding and adds positional information 
# (which will change the meaning of each of the tokens)
positional_embedding = nn.Embedding(BLOCK_SIZE, d_model) # creates the positional embedding space

tok = token_embeddings(x)

# passes into the positional embedding the length of the data input and the device to compute
pos = positional_embedding(
    torch.arange(x.shape[1], device=x.device)
)

h = tok + pos # this is the combined embedding that describes the text, it combines the positional embeddings and the token embeddings together

print(h)


tensor([[[ 0.5266,  2.2050, -0.5778,  ..., -0.2270, -2.2402, -0.8536],
         [-0.8227, -0.0450,  0.1301,  ..., -0.4880, -1.8211, -0.3521],
         [ 0.9028,  3.1035,  0.2094,  ...,  0.3082, -0.9099,  1.3373],
         ...,
         [-1.8099,  0.9698, -0.0573,  ...,  1.6078, -0.8299,  0.2419],
         [ 3.7669, -0.8976,  2.1439,  ...,  0.7458, -3.2934,  0.3925],
         [-0.6133, -2.6274,  1.5697,  ...,  1.4396, -2.5468,  0.5760]],

        [[-1.5178,  1.6372,  1.4632,  ..., -3.3919, -2.9024, -0.6431],
         [-1.8556, -1.4069, -1.3663,  ..., -2.2060, -1.0695,  0.4035],
         [-0.5450,  1.3927, -0.1669,  ...,  0.3523, -0.8639,  0.6936],
         ...,
         [-1.0864,  1.1822,  0.5469,  ...,  0.2782, -0.0657, -1.4957],
         [ 2.9406, -1.4934, -0.7335,  ...,  0.9336, -1.0110, -0.9040],
         [-1.4532, -1.1749,  1.6125,  ...,  1.1508, -1.1236,  2.0027]],

        [[ 0.3741,  1.3047,  1.4401,  ..., -1.5567, -1.6530,  1.0407],
         [ 1.5346,  0.6260, -1.4587,  ...,  0

In [7]:
# Convert the entire dataset into integer token IDs
data = torch.tensor(
    encode(text),
    dtype=torch.long
)
print(data.shape)


# generic training and validation split.
# the standard is usually 90 percent 10 percent because on either ends of the extremes
# with a LARGE amount of data, or a small amount of data, you need to use as much as possible
# in training to maximize performance
split = int(0.9 * len(data))

train_data = data[:split]
val_data = data[split:]

torch.Size([5342509])


In [8]:
# batch making
def get_batch(split_name):
    """
    Returns:

        x: [B, T]
        y: [B, T]

    y is x shifted one character into the future. (next prediction/data)
    """

    source = train_data if split_name == "train" else val_data

    # random starting positions for each sequence
    starts = torch.randint(
        0,
        len(source) - BLOCK_SIZE - 1,
        (batch_size,)
    )

    x = torch.stack([
        source[i:i + BLOCK_SIZE]
        for i in starts
    ])

    y = torch.stack([
        source[i + 1:i + BLOCK_SIZE + 1]
        for i in starts
    ])

    return x.to(device), y.to(device)

In [9]:
# SINGLE ATTENTION HEAD!!!!
class Head(nn.Module):
    def __init__(self, d_model, head_size):
        super().__init__()
        self.key = nn.Linear(d_model, head_size, bias=False)
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)

        # Lower-triangular causal mask
        #
        # Example for T = 4:
        #
        # 1 0 0 0
        # 1 1 0 0
        # 1 1 1 0
        # 1 1 1 1
        # just a fancy matrix for a pretty simple idea

        self.register_buffer(
            "tril", torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE))
        )  # this is very interesting to analyse
        # the purpose of having these 'buffers' is that the model should not be able to look at the future words,
        # and perform calculations based on them. By making the future tokens 0, the model will
        # be unable to allow them to influence current tokens, keeping the attention mechanism
        # one-way. In attention, the past cannot affect the future, until the future occurs
        # then only is the future able to affect the past, and vice-versa. (stops cheating)
        # softmax also resets these back down to 0 after attention makes them -inf

        self.dropout = nn.Dropout(
            dropout
        )  # this is actually also very interesting to analyse
        # essentially, the purpose of Dropout is to prevent overfitting of the data
        # Each node has a probability of being deactivated during training,
        # which ends up decreasing overfitting, and increasing overall model quality
        # A good analogy to this is in the workplace
        # If there are 10 workers who specialize in their own specific tasks, overtime,
        # they are only capable of outputting the very specific things that they were
        # trained to do, which means that they won't be able to challenging or unique problems
        # however, if one of them is sick, and cannot work, than the others pick up
        # some of the info that the missing worker carried, which makes all of them more
        # free-thinking and creative, ultimately making it so that when the sick workers
        # is better, and returns, the other workers are more stimulated, and less hyper-focused
        # on their own specialization, making them more generalized (stopping overfitting)

    """_summary_ This does the forward pass of all the input data. 
    Essentially, when the data gets a linear transformation
    _param_ x = [B, T, C]
    q,k,v = [B, T, H] where H is the head dimension
    
    """

    def forward(self, x):
        B, T, C = x.shape
        
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        
        # computing attention scores
        # it mostly works from using 
        # q: [B, T, H]
        # k.transpose [B, H, T]

        scores = q @ k.transpose(-2, -1)
        
        # this scales scores by sqrt(head_size)
        # The reason for this is very obvious once mathematically explained
        # Essentially, the dot product of vectors is always capped at 1, 
        # however, adding vectors together can increase the magnitude
        # beyond what softmax can effectively control
        # therefore producing unreliable extreme values that prevent the transformer from working
        # The dot product variance is always 1, while the summation effect causes the dot product variation
        # to become it's magnitude
        # dividing sets the scores back down to 1, and therefore keeps the stddev at 1
        scores = scores / (k.shape[-1] ** 0.5)
        
        # applies the casual mask (to prevent it from cheating from the future)
        scores = scores.masked_fill(self.tril[:T, :T] ==0, float('-inf') ) # pyright: ignore[reportIndexIssue]
        
        # SOFTMAX TIME (sexy function ngl)
        # quick recap of softmax is that it tones done extreme values, setting them between 0 and 1
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        
        # this is a weighted combo of all the values
        # essentially multiples the weight matrix and value matrix (Weight * Value)
        out = weights @ v
        
        # [B, T, H]
        return out
        
    

In [10]:
# Multi Head Attention, which is an extension of the current Single Headed one


class MultiHeadedAttention(nn.Module):
    def __init__(self, num_heads, d_model):
        super().__init__()
        # this is extremely important as the num of heads must be a multiple of the model
        # as otherwise the matrix multipliction fails
        assert d_model % num_heads == 0

        # instead of making a separate key, query, and value layer for every head,
        # make all 3 * d_model values in one large GPU matrix multiplication
        # mathematically this is still the exact same idea:
        # X @ [Wq | Wk | Wv] = [X @ Wq | X @ Wk | X @ Wv]
        # the only difference is that the GPU can do the work in one big operation
        self.head_size = d_model // num_heads
        self.num_heads = num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)

        # after concatting the heads, the heads need to be projected back into the d_model,
        # in order to preserve matrix operatioons
        self.projection = nn.Linear(d_model, d_model)

        # as prev explained, strategic disabling of param based on probability (dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        # this single matrix multiplication creates q, k, and v for every head
        # qkv starts as [B, T, 3 * d_model], then each part is [B, T, d_model]
        q, k, v = self.qkv(x).chunk(3, dim=-1)

        # split d_model into the individual heads: [B, T, C] -> [B, num_heads, T, head_size]
        # transpose is important because each head now has its own full T by T attention matrix
        q = q.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_size).transpose(1, 2)

        # mathematically, for each head this is still:
        # softmax((Q @ K.transpose(-2, -1)) / sqrt(head_size) + causal_mask) @ V
        # but on a supported GPU, scaled_dot_product_attention can use a fused CUDA kernel
        # which scales, masks, softmaxes, drops out, and multiplies by V without repeatedly
        # writing the huge [B, num_heads, T, T] score/weight matrices to GPU memory
        # this also removes the Python loop and many tiny GPU kernel launches from above
        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=dropout if self.training else 0.0,
            is_causal=True,
        )

        # combine the heads back into the normal transformer shape [B, T, d_model]
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        out = self.projection(out) 
        out = self.dropout(out)
        return out

In [11]:
# This is the FeedForward layer, which allows for changes to each individual token's understanding and information
# rather than simply being an average of all the sorrounding tokens (which attention will do)
# this instead increases the info that each individual token holds
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        # this is just the neural network, very sexy and compact
        # GeLU is being used rather than ReLU since GeLU is universally better
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model, bias=False),
            nn.GELU(
                approximate="tanh"
            ),  # since tanh is very close to the actual GeLU, it can be used, it's more performance efficient
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model, bias=False),
            nn.Dropout(dropout),
        )

    # very simple forward passing, since each token is only modifying itself, rather than others
    def forward(self, x):
        return self.net(x)

In [12]:
# Transformer block essentially functions as such:
#  attention -> MLP(FeedForwardLayer) -> residual connections -> LayerNorm


class Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.self_attention = MultiHeadedAttention(num_heads=num_heads, d_model=d_model)

        self.ffwd = FeedForward(d_model=d_model)
        self.layer_norm_1 = nn.RMSNorm(
            d_model
        )  # RMSNorm used for performance, LayerNorm is fine

        # by multiplying layer weights by a small amount, the rate at which
        # the model learns is dampened, essentially allowing for smaller adjustment
        # at late-stage
        self.scale_1 = nn.Parameter(torch.ones(d_model) * 1e-2)
        self.scale_2 = nn.Parameter(torch.ones(d_model) * 1e-2)

    # this is also in parallel to allow for slightly faster compute
    def forward(self, x):

        normed_x = self.layer_norm_1(x)

        x = x + self.scale_1 * self.self_attention(normed_x) + self.scale_2 * self.ffwd(normed_x)
        return x

In [13]:
# putting everything together now

class SmallLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embeddings = nn.Embedding(vocab_size, d_model)
        self.positional_embeddings = nn.Embedding(BLOCK_SIZE, d_model)

        # all the transformer layers now

        self.blocks = nn.Sequential(
            *[Block(d_model=d_model, num_heads=num_heads) for i in range(num_layers)]
        )

        # final layer normalization super important for general capability
        self.final_layer_normalization = nn.RMSNorm(d_model)

        # this converts all the embeddings BACK into vocab definitions
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x, targets=None):
        B, T = x.shape

        tokenal_embeddings = self.token_embeddings(x)

        # [B, T, C]
        positions = torch.arange(T, device=x.device)

        # positions shape is [T]

        position_embeddings = self.positional_embeddings(positions)

        # [T,C]
        x = position_embeddings + tokenal_embeddings
        x = self.blocks(x)
        x = self.final_layer_normalization(x)
        vocab_logits = self.lm_head(x)

        # LOSS COMPUTATIONS!!!!!
        # This is genuinly just finding a secant line
        loss = None
        if targets is not None:
            B, T, C = vocab_logits.shape

            # mostly just reshaping matrix sizes
            logits_flat = vocab_logits.reshape(B * T, C)

            targets_flat = targets.reshape(B * T)

            loss = F.cross_entropy(logits_flat, targets_flat)
        return vocab_logits, loss

    # This is autoregressive generation, basically jsut inference
    @torch.no_grad()  # no grad is very helpful in inference as runnign gradient calcualtions take unessesary processing power
    def generate(self, x, max_new_tokens):

        for _ in range(max_new_tokens):

            # Transformer can only handle block_size tokens at once.
            # This is once again because the block_size is the method that is used to splice the text
            x_context = x[:, -BLOCK_SIZE:]

            # forward pass
            logits, _ = self(x_context)

            # Only care about prediction at the final token position
            # this is also where I could add an optional tempreature value, and make no significant change
            # to the general model code
            # Tempreature works by essentially modifying the distributions of the final outputs,
            # ultimately increasing the 'entropy' of the result and therefore
            # making the models outputs more creative
            logits = logits[:, -1, :]

            # [B, vocab_size]

            # Convert logits into probabilities, this is where tempreature can be used
            # also softmax is imporant to keep the entire probability distribution between 0 and 1
            logits /= temperature
            probabilities = F.softmax(logits, dim=-1)

            # smapling next token based on the probabilities (places dice)
            next_token = torch.multinomial(probabilities, num_samples=1)

            # [B, 1]

            # Aadd token to sequence so that the model iterately increases the context
            x = torch.cat((x, next_token), dim=1)

        return x


# the estimate / validation loss (basic concept for gradient descent)
# this helps give an idea as to how good the model is converging onto the correct behaviors
@torch.no_grad()
def estimate_loss(model):

    model.eval()

    results = {}

    for split_name in ["train", "val"]:

        # loss is initally 0, but changes after being run
        losses = torch.zeros(eval_iters)

        for i in range(eval_iters):

            x, y = get_batch(split_name)

            _, loss = model(x, y)

            losses[i] = loss.item()

        results[split_name] = losses.mean().item()

    model.train()

    return results


# now creating the actual model and stuff is pretty straightforward, however an interesting thing
# to note is that the model itself is small enough to be loaded straight onto the GPU memory
# of my laptop, so I noticed that is was very very fast (for not using parallel self-attention computation)
model = SmallLanguageModel().to(device)

num_parameters = sum(p.numel() for p in model.parameters())

print(f"Model parameters: {num_parameters:,}")


# optimizing the model to prevent overfitting
# this is a VERY essential point in making the model perform well
# as without the gold-standard Adam-Weight decay optimizer, the model quickly balloons certain
# weights to be too large and ends up memorizing everything
# AdamW is also a very interesting algorithm, see, AdamW only keeps in 2 pieces of previous data
# which is the momentum (first moment of the weight) (essentially the average direction the weight
# has been moving)
# and then the RMSProp (second moment of the weight) (which finds the magnitude of recent updates)
# This is actually very similar to how algorithms such as PID and even most filters work
# If an LLM is basically a really efficient compression engine, then optimization through AdamW
# is going to be the filter that helps keep the compression engine as effective as possible
# This is very similar to physics through velocity and acceleration
# AdamW will keep the velocity if it has been consistently in a certain direction
# AdamW will dampen the acceleration if the weight is erratic, and amplify the acceleration if it is not moving
# AdamW is basically the great equalizer and judge
# highly highly cool
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# this is training loop

for iteration in range(max_iters):

    # don't want to constantly evaluate, so only done every eval_interval ammount (for sexiness)
    if iteration % eval_interval == 0:

        losses = estimate_loss(model)

        print(
            f"step {iteration:5d} | "
            f"train loss {losses['train']:.4f} | "
            f"val loss {losses['val']:.4f}  |  "
            f"val/train loss ratio {losses['val'] / losses['train']}" # this should ideally stay around 1, any higher is bad (indicating overfitting then)
        )

    # smapling the batch

    x, y = get_batch("train")

    # forward pass

    logits, loss = model(x, y)

    # backprop
    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()


# generate text 
model.eval()



Model parameters: 10,789,332
step     0 | train loss 4.6295 | val loss 4.6221
step   200 | train loss 2.4859 | val loss 2.4990
step   400 | train loss 2.3371 | val loss 2.3601
step   600 | train loss 2.2092 | val loss 2.2173
step   800 | train loss 1.9724 | val loss 1.9876
step  1000 | train loss 1.8187 | val loss 1.8377
step  1200 | train loss 1.7139 | val loss 1.7514
step  1400 | train loss 1.6116 | val loss 1.6612
step  1600 | train loss 1.5512 | val loss 1.6281
step  1800 | train loss 1.4918 | val loss 1.5727
step  2000 | train loss 1.4310 | val loss 1.5190
step  2200 | train loss 1.4078 | val loss 1.5007
step  2400 | train loss 1.3761 | val loss 1.4723
step  2600 | train loss 1.3489 | val loss 1.4638
step  2800 | train loss 1.3124 | val loss 1.4242
step  3000 | train loss 1.3099 | val loss 1.3946
step  3200 | train loss 1.2877 | val loss 1.3981
step  3400 | train loss 1.2709 | val loss 1.3898
step  3600 | train loss 1.2476 | val loss 1.3691
step  3800 | train loss 1.2419 | val los

SmallLanguageModel(
  (token_embeddings): Embedding(84, 384)
  (positional_embeddings): Embedding(256, 384)
  (blocks): Sequential(
    (0): Block(
      (self_attention): MultiHeadedAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=False)
        (projection): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffwd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=False)
          (1): GELU(approximate='tanh')
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=1536, out_features=384, bias=False)
          (4): Dropout(p=0.1, inplace=False)
        )
      )
      (layer_norm_1): RMSNorm((384,), eps=None, elementwise_affine=True)
    )
    (1): Block(
      (self_attention): MultiHeadedAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=False)
        (projection): Linear(in_features=384, out_f

In [ ]:
# since the model needs data to run, assigning first start token is most important
# context = torch.zeros((1, 1), dtype=torch.long, device=device)
custom_text_prompt  = '0'
context = torch.tensor(
    encode(custom_text_prompt),
    dtype=torch.long
)
print(context.shape)
generated = model.generate(context, max_new_tokens=10000)
hashes = generated[0].tolist()
# for i in range(len(hashes)):
#     hashes[i] += (random.random())
#     hashes[i] = int(round(hashes[i]))
    
generated_text = decode(hashes)
print("\n")
print("=" * 60)
print("GENERATED TEXT")
print("=" * 60)
print(generated_text)

torch.Size([1])


IndexError: too many indices for tensor of dimension 1